In [15]:
from pydantic import BaseModel
import uuid

class Task(BaseModel):
    """Task class to hold task information."""
    task_id: str | None = None
    task_status: str = "initiated"
    task_description: str | None = None
    task_name: str
    task_parameters: Dict[str, Any] | None = None
    task_state: Dict[str, Any] | None = None

    def __init__(
            self, 
            task_name: str, 
            task_description = None, 
            task_id: str = None, 
            task_parameters: Dict[str, Any] = None,
            **kwargs
        ) -> None:
        if not task_id:
            task_id = str(uuid.uuid4())
        task_parameters["ext_args"]=kwargs
        task_state=task_parameters["ext_args"].get("task_state",None)
        super().__init__(
            task_id=task_id,
            task_status="pending" if task_id else "initiated",
            task_name=task_name,
            task_description=task_description,
            task_parameters=task_parameters,
        )
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "task_id": self.task_id,
            "task_status": self.task_status,
            "task_name": self.task_name,
            "task_description": self.task_description,
            "task_parameters": self.task_parameters,
            "task_state": self.task_state
        }
    
    def __str__(self) -> str:
        return f"Task: {self.task_name} ({self.task_id}) - {self.task_status}"


In [18]:
task_dict = {"task_name":"extract_data","task_parameters":{"source_content_id":"fc_12345"}}

In [19]:
task = Task(**task_dict)

In [20]:
print(task.task_parameters)

{'source_content_id': 'fc_12345', 'ext_args': {}}


In [1]:
from mcp.server.fastmcp import FastMCP
from autogen_ext.agents.probill import ProbillPlanAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_core.models import ModelFamily
from autogen_agentchat.base import TaskResult


model_info = {
    "vision": False,
    "function_calling": True,
    "json_output": True,
    "family": ModelFamily.R1
}

model_client=OllamaChatCompletionClient(
    model="qwq", 
    host="http://10.0.40.49:11434",
    model_info=model_info,  
)

plan_agent = ProbillPlanAgent("proboll_planner", model_client=model_client, model_client_stream=True)
async def run_stream():
    async for result in plan_agent.run_stream(task="How are you? Don't over thinking."):
        if isinstance(result, TaskResult):
            print("*"*50,"\nFinal Answer\n","*"*50)
            for msg in result.messages:
                print(msg.source)
                print(msg.content)
        else:
            print(result.content)

await run_stream()

TypeError: BaseChatAgent.__init__() got an unexpected keyword argument 'model_client'

In [ ]:
from autogen_core.models import UserMessage
messages = [
    UserMessage(content="Write a very short story about a dragon in Chinese.", source="user"),
]

# Create a stream.
stream = model_client.create_stream(messages=messages)

# Iterate over the stream and print the responses.
print("Streamed responses:")
async for response in stream:  # type: ignore
    if isinstance(response, str):
        # A partial response is a string.
        print(response, flush=True, end="")
    else:
        # The last response is a CreateResult object with the complete message.
        print("\n\n------------\n")
        print("The complete response:", flush=True)
        print(response.content, flush=True)
        print("\n\n------------\n")
        print("The token usage was:", flush=True)
        print(response.usage, flush=True)

Streamed responses:
<think>

Okay, the user wants a very short story about a dragon. Let me think about how to approach this.

First, I need to decide on the key elements of a dragon story. Dragons are often portrayed as mythical creatures with fire-breathing abilities and large size. Maybe I can give my dragon some unique traits to make the story interesting.

Since it's supposed to be short, I should keep the plot simple but engaging. Perhaps focus on a moment or a small adventure rather than an epic journey. Let me consider a setting—maybe a mountain or a cave? 

I want to add some character development for the dragon. Maybe instead of the typical evil dragon guarding treasure, this one has a different motivation. That could make the story more unique. What if the dragon is curious about something outside its usual environment?

Conflict is important even in short stories. The dragon's desire to explore versus staying in its known habitat? Or maybe an unexpected encounter with anoth

In [2]:
_component_plan_agent = plan_agent.dump_component()
component_json = _component_plan_agent.model_dump()

In [3]:
print(component_json)

{'provider': 'autogen_ext.agents.probill.ProbillPlanAgent', 'component_type': 'agent', 'version': 1, 'component_version': 1, 'description': 'An agent, used by Probill that provides plaaning service.', 'label': 'ProbillPlanAgent', 'config': {'name': 'proboll_planner', 'model_client': {'provider': 'autogen_ext.models.ollama.OllamaChatCompletionClient', 'component_type': 'model', 'version': 1, 'component_version': 1, 'description': 'Chat completion client for Ollama hosted models.', 'label': 'OllamaChatCompletionClient', 'config': {'model': 'qwq', 'host': 'http://10.0.40.49:11434', 'follow_redirects': True}}, 'tools': [], 'model_context': {'provider': 'autogen_core.model_context.UnboundedChatCompletionContext', 'component_type': 'chat_completion_context', 'version': 1, 'component_version': 1, 'description': 'An unbounded chat completion context that keeps a view of the all the messages.', 'label': 'UnboundedChatCompletionContext', 'config': {}}, 'description': 'An agent that provides assi

In [4]:
component_json.get("config", {}).get("model_client",{}).get("config")["model_info"]=model_client.model_info

In [5]:
import json
print(json.dumps(component_json, indent=2))

{
  "provider": "autogen_ext.agents.probill.ProbillPlanAgent",
  "component_type": "agent",
  "version": 1,
  "component_version": 1,
  "description": "An agent, used by Probill that provides plaaning service.",
  "label": "ProbillPlanAgent",
  "config": {
    "name": "proboll_planner",
    "model_client": {
      "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
      "component_type": "model",
      "version": 1,
      "component_version": 1,
      "description": "Chat completion client for Ollama hosted models.",
      "label": "OllamaChatCompletionClient",
      "config": {
        "model": "qwq",
        "host": "http://10.0.40.49:11434",
        "follow_redirects": true,
        "model_info": {
          "vision": false,
          "function_calling": true,
          "json_output": true,
          "family": "r1"
        }
      }
    },
    "tools": [],
    "model_context": {
      "provider": "autogen_core.model_context.UnboundedChatCompletionContext",
      "co

In [ ]:
model_cpn = model_client.dump_component()

print(model_cpn.model_dump_json(indent=2))

{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}


In [5]:
model_client.model_info

{'vision': False,
 'function_calling': True,
 'json_output': True,
 'family': 'r1'}

In [20]:
temp = """{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "model_info": {
        "vision": false,
        "function_calling": true,
        "json_output": true,
        "family": "r1"
    },
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}"""

temp_json = json.loads(temp)

In [21]:
print(json.dumps(temp_json, indent=2))

{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "model_info": {
      "vision": false,
      "function_calling": true,
      "json_output": true,
      "family": "r1"
    },
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}


In [ ]:
from autogen_agentchat.teams import MagenticOneGroupChat